In [0]:
%sql
-- ===============================================================================
-- Creating the Gold layer for Streamlit web app deployment
-- ===============================================================================

CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.gold;

USE CATALOG pvdaq_catalog;
USE SCHEMA gold;
SELECT current_catalog(), current_schema();


In [0]:
%pip install pvanalytics
dbutils.library.restartPython()

In [0]:
# =============================================================================
# Gold layer: precompute daily and annual performance metrics per system.
#
# Paste this into a cell in the "PVDAQ Data - Gold" notebook and run it once.
# It moves the pvanalytics math out of the web app and into the pipeline, so
# the Streamlit app only reads a few thousand rows instead of recomputing
# performance ratios from ~2.5M rows of raw sensor data on every page load.
#
# Rerun this cell whenever the Silver layer changes.
# =============================================================================

import pandas as pd
from pvanalytics.metrics import performance_ratio_nrel

spark.sql("CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.gold")

DAYTIME_POA_THRESHOLD = 50  # W/m^2, same cutoff used in the Silver notebook

# Load the full joined time series and the system metadata once
df = (
    spark.table("pvdaq_catalog.silver.pvdata_2020_joined")
    .toPandas()
)
df["utc_measured_on"] = pd.to_datetime(df["utc_measured_on"])
df = df.set_index("utc_measured_on").sort_index()

systems = (
    spark.table("pvdaq_catalog.silver.system")
    .select("system_id", "power", "public_name")
    .toPandas()
)
systems["power"] = pd.to_numeric(systems["power"], errors="coerce")

daily_rows = []
annual_rows = []

for system_id, sys_df in df.groupby("system_id"):

    meta = systems[systems["system_id"] == system_id]
    if meta.empty or pd.isna(meta["power"].iloc[0]):
        print(f"Skipping system {system_id}: no DC capacity in silver.system")
        continue

    # Same normalization as plot_system_pr_and_power(): values above 10,000 are
    # assumed to be watts and converted to kW
    raw_power = float(meta["power"].iloc[0])
    pdc0 = raw_power / 1000.0 if raw_power > 10000 else raw_power

    daytime_full = sys_df[sys_df["poa_irradiance"] >= DAYTIME_POA_THRESHOLD]
    if daytime_full.empty:
        print(f"Skipping system {system_id}: no daytime records")
        continue

    pr_annual = performance_ratio_nrel(
        poa_global=daytime_full["poa_irradiance"],
        temp_air=daytime_full["ambient_temp"],
        wind_speed=daytime_full["wind_speed"],
        pac=daytime_full["ac_power_kw"],
        pdc0=pdc0,
    )

    annual_rows.append(
        {
            "system_id": int(system_id),
            "public_name": str(meta["public_name"].iloc[0]),
            "pdc0_kw": float(pdc0),
            "pr_annual": float(pr_annual),
        }
    )

    for date, day_df in sys_df.groupby(sys_df.index.date):
        daytime_data = day_df[day_df["poa_irradiance"] >= DAYTIME_POA_THRESHOLD]
        if daytime_data.empty:
            continue

        pr = performance_ratio_nrel(
            poa_global=daytime_data["poa_irradiance"],
            temp_air=daytime_data["ambient_temp"],
            wind_speed=daytime_data["wind_speed"],
            pac=daytime_data["ac_power_kw"],
            pdc0=pdc0,
        )

        # Mean over the whole day, including night, matching the notebook
        daily_rows.append(
            {
                "system_id": int(system_id),
                "date": date,
                "pr": float(pr),
                "avg_ac_power_kw": float(day_df["ac_power_kw"].mean()),
            }
        )

    print(f"System {system_id}: annual PR {pr_annual:.3f}, pdc0 {pdc0:.1f} kW")

# -----------------------------------------------------------------------------
# Write the Gold tables
# -----------------------------------------------------------------------------

daily_pdf = pd.DataFrame(daily_rows)
daily_pdf["date"] = pd.to_datetime(daily_pdf["date"])

annual_pdf = pd.DataFrame(annual_rows)

(
    spark.createDataFrame(daily_pdf)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pvdaq_catalog.gold.system_daily_performance")
)

(
    spark.createDataFrame(annual_pdf)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pvdaq_catalog.gold.system_annual_performance")
)

print(f"\nWrote {len(daily_pdf):,} daily rows and {len(annual_pdf)} annual rows.")

In [0]:
# =============================================================================
# Gold layer: precompute daily and annual performance metrics per system.
#
# Paste this into a cell in the "PVDAQ Data - Gold" notebook and run it once.
# It moves the pvanalytics math out of the web app and into the pipeline, so
# the Streamlit app only reads a few thousand rows instead of recomputing
# performance ratios from ~2.5M rows of raw sensor data on every page load.
#
# Rerun this cell whenever the Silver layer changes.
# =============================================================================

import pandas as pd
from pvanalytics.metrics import performance_ratio_nrel

spark.sql("CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.gold")

DAYTIME_POA_THRESHOLD = 50  # W/m^2, same cutoff used in the Silver notebook

# Load the full joined time series and the system metadata once
df = (
    spark.table("pvdaq_catalog.silver.pvdata_2020_joined")
    .toPandas()
)
df["utc_measured_on"] = pd.to_datetime(df["utc_measured_on"])
df = df.set_index("utc_measured_on").sort_index()

systems = (
    spark.table("pvdaq_catalog.silver.system")
    .select("system_id", "power", "public_name")
    .toPandas()
)
systems["power"] = pd.to_numeric(systems["power"], errors="coerce")

daily_rows = []
annual_rows = []

for system_id, sys_df in df.groupby("system_id"):

    meta = systems[systems["system_id"] == system_id]
    if meta.empty or pd.isna(meta["power"].iloc[0]):
        print(f"Skipping system {system_id}: no DC capacity in silver.system")
        continue

    # Same normalization as plot_system_pr_and_power(): values above 10,000 are
    # assumed to be watts and converted to kW
    raw_power = float(meta["power"].iloc[0])
    pdc0 = raw_power / 1000.0 if raw_power > 10000 else raw_power

    daytime_full = sys_df[sys_df["poa_irradiance"] >= DAYTIME_POA_THRESHOLD]
    if daytime_full.empty:
        print(f"Skipping system {system_id}: no daytime records")
        continue

    pr_annual = performance_ratio_nrel(
        poa_global=daytime_full["poa_irradiance"],
        temp_air=daytime_full["ambient_temp"],
        wind_speed=daytime_full["wind_speed"],
        pac=daytime_full["ac_power_kw"],
        pdc0=pdc0,
    )

    annual_rows.append(
        {
            "system_id": int(system_id),
            "public_name": str(meta["public_name"].iloc[0]),
            "pdc0_kw": float(pdc0),
            "pr_annual": float(pr_annual),
        }
    )

    for date, day_df in sys_df.groupby(sys_df.index.date):
        daytime_data = day_df[day_df["poa_irradiance"] >= DAYTIME_POA_THRESHOLD]
        if daytime_data.empty:
            continue

        pr = performance_ratio_nrel(
            poa_global=daytime_data["poa_irradiance"],
            temp_air=daytime_data["ambient_temp"],
            wind_speed=daytime_data["wind_speed"],
            pac=daytime_data["ac_power_kw"],
            pdc0=pdc0,
        )

        # Mean over the whole day, including night, matching the notebook
        daily_rows.append(
            {
                "system_id": int(system_id),
                "date": date,
                "pr": float(pr),
                "avg_ac_power_kw": float(day_df["ac_power_kw"].mean()),
            }
        )

    print(f"System {system_id}: annual PR {pr_annual:.3f}, pdc0 {pdc0:.1f} kW")

# -----------------------------------------------------------------------------
# Write the Gold tables
# -----------------------------------------------------------------------------

daily_pdf = pd.DataFrame(daily_rows)
daily_pdf["date"] = pd.to_datetime(daily_pdf["date"])

annual_pdf = pd.DataFrame(annual_rows)

(
    spark.createDataFrame(daily_pdf)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pvdaq_catalog.gold.system_daily_performance")
)

(
    spark.createDataFrame(annual_pdf)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pvdaq_catalog.gold.system_annual_performance")
)

print(f"\nWrote {len(daily_pdf):,} daily rows and {len(annual_pdf)} annual rows.")